In [1]:
# 1. Montar Drive & caminhos
from google.colab import drive
import os, subprocess

MNT = "/content/drive"
if not os.path.ismount(MNT):
    if os.path.exists(MNT) and os.listdir(MNT):
        try:
            subprocess.run(["fusermount", "-u", MNT], check=False)
        except Exception:
            pass
        !rm -rf /content/drive/*
    drive.mount(MNT, force_remount=True)

ROOT = "MyDrive" if os.path.exists("/content/drive/MyDrive") else \
       ("MeuDrive" if os.path.exists("/content/drive/MeuDrive") else "MyDrive")
BASE_PATH = f"/content/drive/{ROOT}/TCC_USP"
RAW_PATH  = os.path.join(BASE_PATH, "data_raw")
PROC_PATH = os.path.join(BASE_PATH, "data_processed")
os.makedirs(PROC_PATH, exist_ok=True)

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)

Mounted at /content/drive
BASE_PATH: /content/drive/MyDrive/TCC_USP
PROC_PATH: /content/drive/MyDrive/TCC_USP/data_processed


In [2]:
# 2. Carregar dados de entrada
import pandas as pd, os

in_file = os.path.join(PROC_PATH, "news_multisource_clean.parquet")
assert os.path.exists(in_file), f"Arquivo não encontrado: {in_file}. Rode o nb 13 primeiro."

df = pd.read_parquet(in_file)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"]).copy()

print("Shape:", df.shape)
display(df.head(3))

Shape: (100, 6)


,date,title,text,source,url,origin
0,2025-09-19 07:00:00,Ibovespa bate novo recorde e encosta nos 146 m...,"<a href=""https://news.google.com/rss/articles/...",G1,https://news.google.com/rss/articles/CBMieEFVX...,google_rss
1,2025-09-21 07:00:00,Apenas 4 ações do Ibovespa pagam dividendos ac...,"<a href=""https://news.google.com/rss/articles/...",InfoMoney,https://news.google.com/rss/articles/CBMixAFBV...,google_rss
2,2025-09-30 07:00:00,Veja as 10 maiores altas do Ibovespa em setemb...,"<a href=""https://news.google.com/rss/articles/...",Valor Investe,https://news.google.com/rss/articles/CBMi_AFBV...,google_rss


In [3]:
# 3. Instalar & carregar NLP (spaCy PT-BR + fallback)
import sys, importlib

USE_SPACY = True
try:
    import spacy
except Exception:
    !pip -q install spacy==3.7.2
    import spacy

# Tenta carregar o modelo grande pt; se falhar, tenta o pequeno; se falhar, fallback puro
def load_spacy_pt():
    for m in ["pt_core_news_lg", "pt_core_news_md", "pt_core_news_sm"]:
        try:
            return spacy.load(m)
        except Exception:
            try:
                !python -m spacy download {m} -q
                return spacy.load(m)
            except Exception:
                pass
    return None

nlp = load_spacy_pt()
if nlp is None:
    USE_SPACY = False
    print("⚠️ spaCy PT não disponível. Usando fallback simples (regex + NLTK).")
    !pip -q install unidecode
    import re
    from unidecode import unidecode
    import nltk
    nltk.download('stopwords')
    from nltk.corpus import stopwords
    STOP_PT = set(stopwords.words("portuguese"))
else:
    print("✅ spaCy carregado:", nlp.meta.get("name"), nlp.meta.get("version"))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.2/568.2 MB 3.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ spaCy carregado: core_news_lg 3.8.0


In [4]:
# 4. Utilitários de limpeza PT-BR
import re

URL_RE   = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w{2,}\b")
NUM_RE   = re.compile(r"\b\d+(?:[.,]\d+)?\b")
WS_RE    = re.compile(r"\s+")

def normalize_whitespace(t: str) -> str:
    return WS_RE.sub(" ", t).strip()

def build_text(row):
    t = (str(row.get("title", "")) + " " + str(row.get("text", ""))).strip()
    t = URL_RE.sub(" ", t)
    t = EMAIL_RE.sub(" ", t)
    # preserva números? para sentimento, em geral remove:
    t = NUM_RE.sub(" ", t)
    return normalize_whitespace(t)

In [5]:
# 5. Funções de pré-processamento (spaCy ou fallback)
def preprocess_spacy_pt(text: str):
    doc = nlp(text)
    lemmas = []
    for tok in doc:
        if tok.is_space or tok.is_punct or tok.like_url or tok.like_email:
            continue
        if tok.is_stop:
            continue
        lemma = tok.lemma_.lower().strip()
        if not lemma or len(lemma) < 2:
            continue
        # remove só pontuação residual
        if re.fullmatch(r"[^\wáàâãéêíóôõúç]+", lemma, flags=re.IGNORECASE):
            continue
        lemmas.append(lemma)
    clean = " ".join(lemmas)
    return clean, lemmas

def preprocess_fallback_pt(text: str):
    # regex + NLTK + unidecode (sem lematização)
    tx = text.lower()
    tx = URL_RE.sub(" ", tx)
    tx = EMAIL_RE.sub(" ", tx)
    tx = re.sub(r"[^a-zà-úç\s]", " ", tx, flags=re.IGNORECASE)
    tx = normalize_whitespace(tx)
    tokens = [w for w in tx.split() if w not in STOP_PT and len(w) > 1]
    clean = " ".join(tokens)
    return clean, tokens

In [6]:
# 6. Aplicar pré-processamento
from tqdm.auto import tqdm
tqdm.pandas()

df = df.assign(text_full=df.apply(build_text, axis=1))

if USE_SPACY:
    out = df["text_full"].progress_apply(preprocess_spacy_pt)
else:
    out = df["text_full"].progress_apply(preprocess_fallback_pt)

df["clean_text"] = out.apply(lambda x: x[0])
df["lemmas"]     = out.apply(lambda x: x[1])
df["n_tokens"]   = df["lemmas"].apply(len)

# Campos auxiliares
df["day"] = df["date"].dt.floor("D")
print("Pré-processamento concluído. Exemplo:")
display(df[["date","source","title","clean_text","n_tokens"]].head(5))

  0%|          | 0/100 [00:00<?, ?it/s]

Pré-processamento concluído. Exemplo:


,date,source,title,clean_text,n_tokens
0,2025-09-19 07:00:00,G1,Ibovespa bate novo recorde e encosta nos 146 m...,ibovespa bater recorde encosta dólar fechar es...,17
1,2025-09-21 07:00:00,InfoMoney,Apenas 4 ações do Ibovespa pagam dividendos ac...,ação ibovespa pagar dividendo acima cdi selic ...,19
2,2025-09-30 07:00:00,Valor Investe,Veja as 10 maiores altas do Ibovespa em setemb...,grande alta ibovespa setembro investe href= ta...,14
3,2025-09-30 07:00:00,InfoMoney,Ibovespa fecha estável no último pregão do mês...,ibovespa fechar estável pregão setembro subir ...,16
4,2025-09-30 07:00:00,Bora Investir,Fim do pregão viva-voz faz 20 anos e B3 celebr...,pregão viva-voz ano b3 celebrar história expos...,20


In [7]:
# 7. Salvar dataset limpo + BOW diário por fonte
import numpy as np

clean_parquet = os.path.join(PROC_PATH, "news_clean.parquet")
clean_csv     = os.path.join(PROC_PATH, "news_clean.csv")

# dataset limpo
cols_out = ["date","day","source","url","origin","title","text","clean_text","lemmas","n_tokens"]
df_out = df[cols_out].copy()
df_out.to_parquet(clean_parquet, index=False)
df_out.to_csv(clean_csv, index=False, encoding="utf-8")
print("✅ Salvo dataset limpo:\n", clean_parquet, "\n", clean_csv)

# BOW diário por fonte (para TF-IDF no nb 15)
bow = (
    df[["day","source","lemmas"]]
    .explode("lemmas")
    .dropna(subset=["lemmas"])
    .groupby(["day","source","lemmas"], as_index=False)
    .size()
    .rename(columns={"size":"tf"})
)

bow_file = os.path.join(PROC_PATH, "bow_daily.parquet")
bow.to_parquet(bow_file, index=False)
print("✅ Salvo BOW diário por fonte:\n", bow_file)
display(bow.head(10))

✅ Salvo dataset limpo:
 /content/drive/MyDrive/TCC_USP/data_processed/news_clean.parquet 
 /content/drive/MyDrive/TCC_USP/data_processed/news_clean.csv
✅ Salvo BOW diário por fonte:
 /content/drive/MyDrive/TCC_USP/data_processed/bow_daily.parquet


,day,source,lemmas,tf
0,2025-09-19,G1,bater,2
1,2025-09-19,G1,"color=""#6f6f6f"">g1</font",1
2,2025-09-19,G1,dólar,2
3,2025-09-19,G1,encosta,2
4,2025-09-19,G1,estável,1
5,2025-09-19,G1,estável</a>&nbsp;&nbsp;<font,1
6,2025-09-19,G1,fechar,2
7,2025-09-19,G1,g1,1
8,2025-09-19,G1,href=,1
9,2025-09-19,G1,ibovespa,1


In [8]:
# 8. QC rápido (distribuições e amostras)
print("Tamanho do corpus (docs):", df.shape[0])
print("Tokens por doc (p50/p90):",
      int(df["n_tokens"].median()), "/", int(df["n_tokens"].quantile(0.9)))

print("\nTop 15 tokens por origem (amostra):")
top = (
    df[["origin","lemmas"]].explode("lemmas")
      .groupby(["origin","lemmas"]).size()
      .reset_index(name="tf").sort_values(["origin","tf"], ascending=[True, False])
)
for org in top["origin"].unique()[:3]:  # mostra até 3 origens
    subt = top[top["origin"]==org].head(15)
    print(f"\n{org}:")
    print(subt[["lemmas","tf"]].to_string(index=False))
print("\nFeito ✅")

Tamanho do corpus (docs): 100
Tokens por doc (p50/p90): 19 / 24

Top 15 tokens por origem (amostra):

google_rss:
                         lemmas  tf
                          href= 100
                       ibovespa  80
                             b3  74
                          dólar  49
       target="_blank">ibovespa  48
                         fechar  42
                             r$  42
                           cair  38
color="#6f6f6f">infomoney</font  27
                      infomoney  27
           /a>&nbsp;&nbsp;<font  25
                          subir  25
                           ação  22
                         recuar  22
                           alta  21

Feito ✅
